<a href="https://colab.research.google.com/github/ghalde194/APP1/blob/main/demitidos_datasul_Excel_o_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [97]:
from google.colab import files
import pandas as pd
from datetime import datetime

print("🚀 PROGRAMA 2 - Demitidos da Datasul que ainda estão no Workday")
print("=" * 85)

# Upload do arquivo
uploaded = files.upload()
nome_arquivo = next(iter(uploaded))

# === DIAGNÓSTICO E SELEÇÃO DE ABAS ===
xls = pd.ExcelFile(nome_arquivo)
print("\n📋 ABAS ENCONTRADAS NO ARQUIVO:")
for i, aba in enumerate(xls.sheet_names, 1):
    print(f"{i}. '{aba}'")

# Busca automática pelas abas (mais tolerante)
aba_datasul = None
aba_workday = None

for aba in xls.sheet_names:
    if "datasul" in aba.lower():
        aba_datasul = aba
    if "workday" in aba.lower():
        aba_workday = aba

if not aba_datasul:
    print("\n❌ Não encontrei nenhuma aba com 'Datasul'.")
    print("Verifique o nome exato acima e me avise.")
    raise Exception("Aba Datasul não encontrada")

if not aba_workday:
    print("\n❌ Não encontrei nenhuma aba com 'Workday'.")
    print("Verifique o nome exato acima e me avise.")
    raise Exception("Aba Workday não encontrada")

print(f"\n✅ Usando aba: '{aba_datasul}' como Datasul")
print(f"✅ Usando aba: '{aba_workday}' como Workday")

# Ler as abas usando os nomes detectados
df_datasul = pd.read_excel(nome_arquivo, sheet_name=aba_datasul)
df_workday = pd.read_excel(nome_arquivo, sheet_name=aba_workday)

print(f"✅ Datasul carregada: {len(df_datasul)} linhas")
print(f"✅ Workday carregada: {len(df_workday)} linhas")

# Padronização da Matrícula
df_datasul["Matrícula"] = df_datasul["Matrícula"].astype(str).str.strip()
df_workday["ID DATASUL"] = df_workday["ID DATASUL"].astype(str).str.strip()

# ==================== IDENTIFICAR DEMITIDOS ====================
# Procurar a coluna "Data Desligamento"
col_desligamento = None
for col in df_datasul.columns:
    if "desligamento" in col.lower():
        col_desligamento = col
        break

if col_desligamento:
    print(f"✅ Coluna encontrada: '{col_desligamento}'")

    # Filtrar funcionários com Data Desligamento preenchida
    demitidos = df_datasul[
        df_datasul[col_desligamento].notna() &
        (df_datasul[col_desligamento].astype(str).str.strip() != "")
    ].copy()

    print(f"Total de funcionários com Data Desligamento: {len(demitidos)}")
else:
    print("❌ Não encontrei a coluna 'Data Desligamento'.")
    demitidos = pd.DataFrame()

# Verificar quais desses demitidos ainda estão no Workday
if len(demitidos) > 0:
    demitidos_no_workday = demitidos[
        demitidos["Matrícula"].isin(df_workday["ID DATASUL"])
    ].copy()
else:
    demitidos_no_workday = pd.DataFrame()

print("\n" + "="*75)
print("RESULTADO FINAL")
print("="*75)
print(f"Total de demitidos na Datasul                  : {len(demitidos)}")
print(f"🎯 Demitidos que ainda estão no Workday        : {len(demitidos_no_workday)}")

if len(demitidos_no_workday) > 0:
    print(f"\n⚠️  ALERTA: {len(demitidos_no_workday)} funcionários demitidos ainda aparecem no Workday!")
else:
    print("\n✅ Nenhum funcionário demitido da Datasul está presente no Workday.")

# ====================== SALVAR ======================
nome_saida = f"Demitidos_no_Workday_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

with pd.ExcelWriter(nome_saida, engine="openpyxl") as writer:
    if len(demitidos_no_workday) > 0:
        demitidos_no_workday.to_excel(writer, sheet_name="Demitidos ainda no Workday", index=False)
    else:
        pd.DataFrame({"Status": ["Nenhum demitido encontrado no Workday"]}).to_excel(
            writer, sheet_name="Demitidos ainda no Workday", index=False)

    # Formatação profissional
    ws = writer.sheets["Demitidos ainda no Workday"]
    for col in ws.columns:
        max_length = 0
        column_letter = col[0].column_letter
        for cell in col:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        ws.column_dimensions[column_letter].width = min(max_length + 3, 45)
    ws.freeze_panes = "A2"

print(f"\n✅ Arquivo gerado: {nome_saida}")
files.download(nome_saida)

🚀 PROGRAMA 2 - Demitidos da Datasul que ainda estão no Workday


Saving Relatórios Funcionários 300326 - Leonardo.xlsx to Relatórios Funcionários 300326 - Leonardo (78).xlsx

📋 ABAS ENCONTRADAS NO ARQUIVO:
1. 'Datasul '
2. 'Workday'
3. 'Atividade'

✅ Usando aba: 'Datasul ' como Datasul
✅ Usando aba: 'Workday' como Workday
✅ Datasul carregada: 869 linhas
✅ Workday carregada: 839 linhas
✅ Coluna encontrada: 'Data Desligamento'
Total de funcionários com Data Desligamento: 26

RESULTADO FINAL
Total de demitidos na Datasul                  : 26
🎯 Demitidos que ainda estão no Workday        : 0

✅ Nenhum funcionário demitido da Datasul está presente no Workday.

✅ Arquivo gerado: Demitidos_no_Workday_20260331_1614.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>